In [1]:
!python -m pip install pandas numpy matplotlib seaborn openpyxl

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [9]:
rider = pd.read_excel("riders.xlsx")
driver = pd.read_excel("drivers.xlsx")

rider.head()

,id,request_time,pickup_time,dropoff_time,pickup_location,dropoff_location,status,request_datetime,pickup_datetime,dropoff_datetime
0,1,5.048803,5.372603,5.668817,"(8.52776815255513, 11.374422951736026)","(11.22406718305943, 18.28586415472845)",dropped-off,2025-12-01 13:02:55.690,2025-12-01 13:22:21.370,2025-12-01 13:40:07.743
1,2,5.113495,5.344117,5.658663,"(4.7260253234018075, 13.921168220313202)","(11.219433016467919, 9.224932723238288)",dropped-off,2025-12-01 13:06:48.580,2025-12-01 13:20:38.822,2025-12-01 13:39:31.187
2,3,5.114319,5.819045,6.154474,"(9.34931432850313, 14.855954034601385)","(15.818906976292498, 14.556629597532137)",dropped-off,2025-12-01 13:06:51.547,2025-12-01 13:49:08.563,2025-12-01 14:09:16.107
3,4,5.210384,5.492225,5.915581,"(5.341657715440439, 9.121586876730834)","(11.832789025965816, 11.90864135355879)",dropped-off,2025-12-01 13:12:37.383,2025-12-01 13:29:32.010,2025-12-01 13:54:56.092
4,5,5.229512,5.421566,6.019653,"(5.444067188695589, 13.774420071355813)","(11.208517457838962, 5.792638826367501)",dropped-off,2025-12-01 13:13:46.244,2025-12-01 13:25:17.639,2025-12-01 14:01:10.751


In [8]:
driver.head()

,id,arrival_time,offline_time,initial_location,current_location,status,arrival_datetime,offline_datetime
0,1,5.011975,11.638223,"(0.7897740952818868, 8.194226996886005)","(13.492895938215721, 17.375090044074234)",offline,2025-12-01 13:00:43.110,2025-12-01 19:38:17.601
1,2,5.557763,11.732854,"(4.600342938200451, 13.82151625336585)","(13.6809497478333, 15.070060581064372)",offline-scheduled,2025-12-01 13:33:27.947,2025-12-01 19:43:58.273
2,3,5.661047,12.723458,"(9.09829206926925, 11.669336718393772)","(15.86922281798255, 9.936190354974855)",offline-scheduled,2025-12-01 13:39:39.768,2025-12-01 20:43:24.450
3,4,5.757413,12.793147,"(14.723720219958555, 14.743429470057318)","(14.232702333916698, 12.939560071816741)",offline-scheduled,2025-12-01 13:45:26.686,2025-12-01 20:47:35.330
4,5,5.769205,12.431118,"(13.023825947458846, 12.501871643860605)","(19.845683334071563, 7.3885004325174215)",offline-scheduled,2025-12-01 13:46:09.139,2025-12-01 20:25:52.025


In [4]:
rider.info()
driver.info()

<class 'pandas.DataFrame'>
RangeIndex: 34421 entries, 0 to 34420
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   id                34421 non-null  int64         
 1   request_time      34421 non-null  float64       
 2   pickup_time       34421 non-null  float64       
 3   dropoff_time      34421 non-null  float64       
 4   pickup_location   34421 non-null  str           
 5   dropoff_location  34421 non-null  str           
 6   status            34421 non-null  str           
 7   request_datetime  34421 non-null  datetime64[us]
 8   pickup_datetime   34142 non-null  datetime64[us]
 9   dropoff_datetime  34134 non-null  datetime64[us]
dtypes: datetime64[us](3), float64(3), int64(1), str(3)
memory usage: 2.6 MB
<class 'pandas.DataFrame'>
RangeIndex: 4719 entries, 0 to 4718
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            -----

In [5]:
rider = rider.sort_values("request_time")
rider["interarrival"] = rider["request_time"].diff()

mean_interarrival = rider["interarrival"].mean()
rider_arrival_rate = 1 / mean_interarrival

print("Mean interarrival (hours):", mean_interarrival)
print("Estimated rider arrival rate (per hour):", rider_arrival_rate)

Mean interarrival (hours): 0.028904982877070474
Estimated rider arrival rate (per hour): 34.59611113602397


In [6]:
driver = driver.sort_values("arrival_time")
driver["interarrival"] = driver["arrival_time"].diff()

mean_driver_interarrival = driver["interarrival"].mean()
driver_arrival_rate = 1 / mean_driver_interarrival

print("Estimated driver arrival rate (per hour):", driver_arrival_rate)

Estimated driver arrival rate (per hour): 4.742253961680111


In [10]:
driver["availability_duration"] = (
    driver["offline_time"] - driver["arrival_time"]
)

driver["availability_duration"].describe()

count    4719.000000
mean        7.004658
std         0.576104
min         6.000736
25%         6.505672
50%         7.016763
75%         7.501668
max         7.999785
Name: availability_duration, dtype: float64

In [12]:
rider["status"].unique()

<StringArray>
['dropped-off', 'abandoned', 'dropoff-scheduled', 'pickup-scheduled']
Length: 4, dtype: str

In [11]:
completed = rider[rider["status"] == "dropped-off"].copy()

completed["waiting_time"] = (
    completed["pickup_time"] - completed["request_time"]
)

completed["waiting_time"].describe()

count    34117.000000
mean         0.236678
std          0.177407
min          0.000437
25%          0.105449
50%          0.190927
75%          0.322548
max          1.419922
Name: waiting_time, dtype: float64

In [13]:
abandonment_rate = (rider["status"] == "abandoned").mean()
print("Abandonment rate:", abandonment_rate)

Abandonment rate: 0.008105516980912815


In [15]:
completed["waiting_time"].quantile([0.5, 0.75, 0.9, 0.95])

0.50    0.190927
0.75    0.322548
0.90    0.480379
0.95    0.590573
Name: waiting_time, dtype: float64

In [16]:
driver["availability_duration"].min(), driver["availability_duration"].max()

(np.float64(6.000735775974306), np.float64(7.999785397523965))